In [2]:
# --- Control Flags ---
CREATE_BEV_ANIMATION = False        # Create and display the BEV GT box animation
GENERATE_GT_LABEL_PYTORCH = True    # Generate the scene-level .pt files with point labels 
GENERATE_ALL_LABELS = True          # If GENERATE_GT_LABEL_PYTORCH is true, whether to generate labels for all scenes
DISPLAY_K3D_VISUALIZATION = False   # Display the K3D plot for a specific sweep

# --- Imports ---
import os
import sys
from nuscenes.nuscenes import NuScenes
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import numpy as np # For K3D example data if NPZs are missing

# --- Add Project Root to sys.path ---
NOTEBOOK_DIR = os.path.abspath('')
# Assuming the notebook is in a 'notebooks' directory at the same level as 'src', 'config', etc.
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR) 
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(f"Added project root to sys.path: {PROJECT_ROOT}")

# --- Import Custom Modules (after adding project root) ---
from src.config_loader import MDetectorConfigAccessor
from src.utils.label_generation import (
    generate_and_save_point_labels_for_scene_pytorch,
    find_instances_in_scene,
    get_scene_sweep_data_sequence
)
# from src.visualization.animation_helpers import create_synchronized_animation
# from src.visualization.k3d_visualizer import visualize_sweep_k3d

# --- Load Configuration using Accessor --- 
try:
    config_file_path = os.path.join(PROJECT_ROOT, 'config/m_detector_config.yaml')
    config_accessor = MDetectorConfigAccessor(config_file_path)
    print(f"Configuration loaded successfully using MDetectorConfigAccessor from: {config_file_path}")
except FileNotFoundError:
    print(f"ERROR: Config file not found at {config_file_path}. Please check the path.")
    config_accessor = None 
except Exception as e:
    print(f"Error loading config with MDetectorConfigAccessor: {e}")
    config_accessor = None

# --- Initialize NuScenes ---
nusc = None
if config_accessor:
    nuscenes_params = config_accessor.get_nuscenes_params()
    try:
        nusc = NuScenes(
            version=nuscenes_params.get('version'),
            dataroot=nuscenes_params.get('dataroot'),
            verbose=nuscenes_params.get('verbose_init', False)
        )
        print(f"NuScenes SDK initialized (Version: {nuscenes_params.get('version')}).")
    except Exception as e:
        print(f"ERROR: NuScenes initialization failed: {e}. Check 'version' and 'dataroot' in config.")
else:
    print("ERROR: MDetectorConfigAccessor failed to initialize. Cannot proceed with NuScenes SDK.")

# --- Select Scene for Demonstration ---
my_scene_rec = None
if nusc and config_accessor:
    # Get default_scene_index from video_generation_params for consistency, or use a notebook-specific default
    video_gen_params = config_accessor.get_video_generation_params()
    target_scene_index = video_gen_params.get('default_scene_index', 1) # Example: default to index 1

    if 0 <= target_scene_index < len(nusc.scene):
        my_scene_rec = nusc.scene[target_scene_index]
        print(f"Target Scene for demo: '{my_scene_rec['name']}' (Index: {target_scene_index})")
        print(f"Description: {my_scene_rec['description']}")
    else:
        print(f"Could not select scene with index {target_scene_index}. Check NuScenes initialization and scene index.")
elif not nusc:
    print("NuScenes SDK not initialized. Cannot select a scene.")
elif not config_accessor:
    print("Config accessor not initialized. Cannot determine target scene index.")

Configuration loaded successfully using MDetectorConfigAccessor from: /home/drugge/Unsupervised-Moving-Point-Detection/m_detector_python/config/m_detector_config.yaml
NuScenes SDK initialized (Version: v1.0-mini).
Target Scene for demo: 'scene-0103' (Index: 1)
Description: Many peds right, wait for turning car, long bike rack left, cyclist


In [3]:
if CREATE_BEV_ANIMATION and nusc and my_scene_rec and config_accessor: # MODIFIED: Added config_accessor check
    print(f"\n--- Preparing for BEV Animation of GT Boxes for scene: {my_scene_rec['name']} ---")
    
    all_sweep_data_for_animation = list(get_scene_sweep_data_sequence(nusc, my_scene_rec['token']))
    
    if not all_sweep_data_for_animation:
        print(f"No LiDAR sweeps found for scene {my_scene_rec['name']}. Skipping BEV animation.")
    else:
        instances_tokens_for_anim = find_instances_in_scene(nusc, my_scene_rec['token'], min_annotations=1)
        
        if not instances_tokens_for_anim:
            print(f"No instances with sufficient annotations found in scene {my_scene_rec['name']}. Skipping BEV animation.")
        else:
            print(f"Found {len(instances_tokens_for_anim)} instances to animate across {len(all_sweep_data_for_animation)} sweeps.")
            
            plt.rcParams['animation.embed_limit'] = 2**128
            print("Generating BEV animation. This may take a few minutes...")
            
            # MODIFIED: Get animation settings using config_accessor
            anim_params = config_accessor.get_visualization_params().get('animation_settings', {})
            
            # Determine save_path for animation if specified in config
            output_paths_cfg = config_accessor.get_mdetector_output_paths()
            anim_save_dir = output_paths_cfg.get('animations_directory', os.path.join(PROJECT_ROOT, 'output/animations'))
            os.makedirs(anim_save_dir, exist_ok=True)
            anim_filename = anim_params.get('bev_gt_animation_filename_template', 'gt_boxes_bev_scene_{scene_name}.mp4').format(scene_name=my_scene_rec['name'])
            # Set save_path to None if 'save_bev_gt_animation' is false or filename is empty
            save_path_for_anim = None
            if anim_params.get('save_bev_gt_animation', False) and anim_filename:
                 save_path_for_anim = os.path.join(anim_save_dir, anim_filename)


            animation_html, animation_object = create_synchronized_animation(
                nusc=nusc,
                instance_tokens=instances_tokens_for_anim,
                all_sweep_data_dicts=all_sweep_data_for_animation,
                scene_name=my_scene_rec['name'],
                interval_ms=anim_params.get('interval_ms', 100),
                figsize=tuple(anim_params.get('figsize', (10, 10))),
                point_downsample=anim_params.get('point_downsample', 5),
                save_path=save_path_for_anim, # MODIFIED: Use determined save_path
                save_writer=anim_params.get('save_writer', 'ffmpeg'),
                save_fps=anim_params.get('save_fps', 10),
                save_dpi=anim_params.get('save_dpi', 150)
            )
            print("Done!")
            
            if animation_html:
                print("Preparing BEV animation for display...")
                display(HTML(animation_html) if isinstance(animation_html, str) else animation_html)
            elif save_path_for_anim and animation_object: # Check if save_path was set
                print(f"BEV Animation saved to: {save_path_for_anim}")
            else:
                print("BEV Animation generation failed or no output produced (and not saved).")
else:
    if not CREATE_BEV_ANIMATION: print("\nSkipping BEV Animation generation as per flag.")
    if not (nusc and my_scene_rec): print("\nSkipping BEV Animation due to NuScenes/Scene initialization issues.")
    if not config_accessor: print("\nSkipping BEV Animation due to config accessor initialization issues.")


Skipping BEV Animation generation as per flag.


In [4]:
if GENERATE_GT_LABEL_PYTORCH and nusc and config_accessor: 
    print(f"\n--- Preparing for Sparse GT Point Label PyTorch (.pt) File Generation ---") 
    
    # Get the single velocity threshold from your config
    validation_params = config_accessor.get_validation_params()
    gt_vel_thresh = validation_params.get('gt_velocity_threshold')
    print(f"Using a single GT velocity threshold for all files: {gt_vel_thresh:.2f} m/s")

    nuscenes_params = config_accessor.get_nuscenes_params()
    gt_labels_output_dir = nuscenes_params.get('label_path')

    if not gt_labels_output_dir:
        print("ERROR: 'nuscenes.label_path' not defined in config.")
    else:
        # Make path absolute
        if not os.path.isabs(gt_labels_output_dir):
            gt_labels_output_dir = os.path.join(PROJECT_ROOT, gt_labels_output_dir)
        
        # --- IMPORTANT: Clean up old files ---
        print(f"Clearing old GT label files from: {gt_labels_output_dir}")
        if os.path.exists(gt_labels_output_dir):
            for f in os.listdir(gt_labels_output_dir):
                if f.endswith('.pt'):
                    os.remove(os.path.join(gt_labels_output_dir, f))
        os.makedirs(gt_labels_output_dir, exist_ok=True)
        # --- END CLEANUP ---

        if not GENERATE_ALL_LABELS:
            # Logic for a single scene
            print(f"\nStarting sparse GT label generation for scene: {my_scene_rec['name']}...")
            generate_and_save_point_labels_for_scene_pytorch(
                nusc=nusc, 
                scene_token=my_scene_rec['token'], 
                output_dir=gt_labels_output_dir,
                gt_velocity_threshold=gt_vel_thresh,
                verbose=True
            )
        else:
            # Logic for all scenes
            print("\nStarting sparse GT point label generation for ALL scenes...")
            for scene_idx in range(len(nusc.scene)):
                current_scene = nusc.scene[scene_idx]
                print(f"\n--- Processing scene {scene_idx + 1}/{len(nusc.scene)}: {current_scene['name']} ---")
                
                generate_and_save_point_labels_for_scene_pytorch(
                    nusc=nusc, 
                    scene_token=current_scene['token'], 
                    output_dir=gt_labels_output_dir,
                    gt_velocity_threshold=gt_vel_thresh,
                    verbose=True
                )
            print(f"\nSparse GT point label generation for ALL scenes complete.")
else:
    if not GENERATE_GT_LABEL_PYTORCH: print("\nSkipping GT Label .pt generation as per flag.")
    if not nusc: print("\nSkipping GT Label HDF5 generation due to NuScenes initialization issues.")
    if not config_accessor: print("\nSkipping GT Label HDF5 generation due to config accessor initialization issues.")


--- Preparing for Sparse GT Point Label PyTorch (.pt) File Generation ---
Using a single GT velocity threshold for all files: 1.00 m/s
Clearing old GT label files from: /home/drugge/staff-umbrella/TeamHolgerResearch/drugge/nuscenes_labels_interpolated

Starting sparse GT point label generation for ALL scenes...

--- Processing scene 1/10: scene-0061 ---
Processing scene for sparse GT labels: scene-0061 (Vel Threshold: 1.0 m/s)


  Generating Sparse GT Labels: 100%|██████████| 382/382 [00:05<00:00, 74.21it/s] 


Successfully saved sparse GT data to /home/drugge/staff-umbrella/TeamHolgerResearch/drugge/nuscenes_labels_interpolated/gt_sparse_labels_scene-0061_v1.0.pt

--- Processing scene 2/10: scene-0103 ---
Processing scene for sparse GT labels: scene-0103 (Vel Threshold: 1.0 m/s)


  Generating Sparse GT Labels: 100%|██████████| 389/389 [00:04<00:00, 84.01it/s] 


Successfully saved sparse GT data to /home/drugge/staff-umbrella/TeamHolgerResearch/drugge/nuscenes_labels_interpolated/gt_sparse_labels_scene-0103_v1.0.pt

--- Processing scene 3/10: scene-0553 ---
Processing scene for sparse GT labels: scene-0553 (Vel Threshold: 1.0 m/s)


...

Successfully saved sparse GT data to /home/drugge/staff-umbrella/TeamHolgerResearch/drugge/nuscenes_labels_interpolated/gt_sparse_labels_scene-1077_v1.0.pt

--- Processing scene 9/10: scene-1094 ---
Processing scene for sparse GT labels: scene-1094 (Vel Threshold: 1.0 m/s)


  Generating Sparse GT Labels: 100%|██████████| 391/391 [00:05<00:00, 70.80it/s] 


Successfully saved sparse GT data to /home/drugge/staff-umbrella/TeamHolgerResearch/drugge/nuscenes_labels_interpolated/gt_sparse_labels_scene-1094_v1.0.pt

--- Processing scene 10/10: scene-1100 ---
Processing scene for sparse GT labels: scene-1100 (Vel Threshold: 1.0 m/s)


  Generating Sparse GT Labels: 100%|██████████| 391/391 [00:02<00:00, 144.38it/s]

Successfully saved sparse GT data to /home/drugge/staff-umbrella/TeamHolgerResearch/drugge/nuscenes_labels_interpolated/gt_sparse_labels_scene-1100_v1.0.pt

Sparse GT point label generation for ALL scenes complete.


In [ ]:
if DISPLAY_K3D_VISUALIZATION and nusc and my_scene_rec and config_accessor: # MODIFIED: Added config_accessor check
    print(f"\n--- Preparing for K3D Visualization ---")
    
    scene_to_visualize_token = my_scene_rec['token']
    scene_to_visualize_name = my_scene_rec['name']
    
    sweeps_in_scene_for_k3d = list(get_scene_sweep_data_sequence(nusc, scene_to_visualize_token))
    sweep_index_to_visualize = 4 
    
    target_lidar_sd_token_for_k3d = None
    if sweeps_in_scene_for_k3d and 0 <= sweep_index_to_visualize < len(sweeps_in_scene_for_k3d):
        target_lidar_sd_token_for_k3d = sweeps_in_scene_for_k3d[sweep_index_to_visualize]['lidar_sd_token']
        print(f"Selected for K3D: Scene '{scene_to_visualize_name}', Sweep Index {sweep_index_to_visualize}, Token: ...{target_lidar_sd_token_for_k3d[-8:]}")
    else:
        print(f"Could not select sweep at index {sweep_index_to_visualize} for scene '{scene_to_visualize_name}'. Skipping K3D plot.")

    if target_lidar_sd_token_for_k3d:
        # MODIFIED: Get paths using config_accessor and expect HDF5
        nuscenes_params = config_accessor.get_nuscenes_params()
        mdet_output_paths = config_accessor.get_mdetector_output_paths()

        gt_labels_hdf5_dir_k3d = nuscenes_params.get('label_path', 'output/gt_point_labels_hdf5') # Default if not in config
        if not os.path.isabs(gt_labels_hdf5_dir_k3d):
             gt_labels_hdf5_dir_k3d = os.path.join(PROJECT_ROOT, gt_labels_hdf5_dir_k3d)
        gt_labels_hdf5_file_k3d = os.path.join(gt_labels_hdf5_dir_k3d, f"gt_point_labels_{scene_to_visualize_name}.h5") # MODIFIED: .h5

        mdet_results_dir_k3d = mdet_output_paths.get('save_path', 'output/mdetector_results') # Default if not in config
        if not os.path.isabs(mdet_results_dir_k3d):
            mdet_results_dir_k3d = os.path.join(PROJECT_ROOT, mdet_results_dir_k3d)
        mdet_results_hdf5_file_k3d = os.path.join(mdet_results_dir_k3d, f"mdet_results_{scene_to_visualize_name}.h5") # MODIFIED: .h5
        
        if not os.path.exists(gt_labels_hdf5_file_k3d):
            print(f"WARNING: GT labels HDF5 file not found for K3D: {gt_labels_hdf5_file_k3d}.")
            gt_labels_hdf5_file_k3d = None
            
        if not os.path.exists(mdet_results_hdf5_file_k3d):
            print(f"INFO: M-Detector results HDF5 file not found for K3D: {mdet_results_hdf5_file_k3d}.")
            mdet_results_hdf5_file_k3d = None

        # MODIFIED: Get k3d_plot settings using config_accessor
        k3d_plot_params = config_accessor.get_k3d_plot_params()

        print("Generating K3D plot...")
        k3d_plot_object = visualize_sweep_k3d(
            nusc=nusc,
            scene_token=scene_to_visualize_token,
            target_sweep_lidar_sd_token=target_lidar_sd_token_for_k3d,
            gt_labels_hdf5_path=gt_labels_hdf5_file_k3d,         # MODIFIED: Pass HDF5 path
            mdet_results_hdf5_path=mdet_results_hdf5_file_k3d, # MODIFIED: Pass HDF5 path
            config_accessor=config_accessor,                     # MODIFIED: Pass accessor
            show_gt_boxes=k3d_plot_params.get('show_gt_boxes', True),
            show_gt_points=k3d_plot_params.get('show_gt_points', True),
            show_mdet_points=k3d_plot_params.get('show_mdet_points', True),
            point_size=k3d_plot_params.get('point_size', 0.05),
            downsample_factor=k3d_plot_params.get('downsample_factor', 1)
        )

        if k3d_plot_object:
            print("Displaying K3D plot...")
            display(k3d_plot_object)
        else:
            print("K3D plot generation failed or no plot object returned.")
else:
    if not DISPLAY_K3D_VISUALIZATION: print("\nSkipping K3D Visualization as per flag.")
    if not (nusc and my_scene_rec): print("\nSkipping K3D Visualization due to NuScenes/Scene initialization issues.")
    if not config_accessor: print("\nSkipping K3D Visualization due to config accessor initialization issues.")